# DungeonEscape: evaluate a decentralized cooperative policy

This notebook downloads the published Causal GPT-RL ONNX policy and its matching model-removed Unity build, inspects the fixed-batch contract, and measures individual and cooperative-group return. ONNX conversion is not required or covered here.

## What is being evaluated?

DungeonEscape has three cooperative agents per match. Training data stores one ego-centric trajectory per agent and links the three trajectories with `match_id`/`group_id`. At inference, the same shared policy runs once per agent. The 36-row ONNX graph evaluates 12 independent arenas × 3 agents in one call; batch rows do not communicate or attend to one another.

In [ ]:
%pip install -q -r examples/unity_collection/requirements-collect.txt huggingface_hub

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

root = Path("hf_unity")
policy = Path(hf_hub_download(
    repo_id="ccnets/causal-gpt-rl-unity",
    filename="dungeon-escape/dungeonescape-b36.onnx",
    local_dir=root / "model",
))

# The build is published independently from the policy. If it is not yet
# present, point `build` below at a compatible local release-23 build.
snapshot_download(
    repo_id="ccnets/causal-gpt-rl-unity-envs",
    repo_type="dataset",
    allow_patterns="DungeonEscape/**",
    local_dir=root / "envs",
)
build = root / "envs/DungeonEscape/UnityEnvironment.exe"
policy, build

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(str(policy), providers=["CPUExecutionProvider"])
print("inputs")
for item in session.get_inputs():
    print(f"  {item.name:8s} {item.shape} {item.type}")
print("outputs")
for item in session.get_outputs():
    print(f"  {item.name:8s} {item.shape} {item.type}")

Expected contract: `states[36,32,371]`, `actions[36,32,7]`, `is_bos[36,32,1]`, `mask[36,32]` → `action[36,7]`. Each output row contains Discrete(7) logits; use `argmax`, then feed a one-hot action back into that row's autoregressive window.

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "examples/unity/evaluate_onnx.py",
    "--build", str(build),
    "--onnx", str(policy),
    "--time-scale", "20",
], check=True)

## Reference data

The training dataset contains 35,787 agent episodes with mean agent return **0.606589**. The 11,929 linked three-agent matches have mean summed group return **1.819767** and group success rate **95.32%**.

A zero-return agent is not automatically a failed match: it may terminate before surviving teammates unlock the door. Compare the evaluator's `[agent return]` and `[cooperative group return]` blocks to the corresponding references.

The published ONNX is an **intermediate training checkpoint** provided to validate the end-to-end download and evaluation workflow. Its reported performance is provisional and does not represent the final trained policy. The artifact and metrics will be updated after training completes.